# SPARK 2026 Mini Challenge
## Team Zambia - Chest X-Ray Pneumonia Classification

### Task
Build a CNN from scratch in PyTorch to classify chest X-rays as:
- **0** = Normal
- **1** = Pneumonia

### Evaluation Metric
Macro-averaged F1-score

### Dataset Citation
Kermany, D., Goldbaum, M., Cai, W. et al. (2018).
Identifying Medical Diagnoses and Treatable Diseases by Image-Based Deep Learning.
Cell, 172(5), 1122-1131.
https://doi.org/10.1016/j.cell.2018.02.010

In [16]:
# SPARK 2026 Mini Challenge
# Team Zambia - Chest X-Ray Pneumonia Classification
#
# Task:
# Build a CNN from scratch in PyTorch to classify chest X-rays as:
# 0 = Normal
# 1 = Pneumonia
#
# Evaluation metric:
# Macro-averaged F1-score
#
# Dataset citation:
# Kermany, D., Goldbaum, M., Cai, W. et al. (2018).
# Identifying Medical Diagnoses and Treatable Diseases by Image-Based Deep Learning.
# Cell, 172(5), 1122-1131.
# https://doi.org/10.1016/j.cell.2018.02.010

## Cell 2 — Imports

In [17]:
import os
import random
import numpy as np
import pandas as pd
from PIL import Image

import torch
import torch.nn as nn
import torch.optim as optim

from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from torchvision import transforms

from sklearn.metrics import f1_score, confusion_matrix, classification_report

## Cell 3 — Reproducibility

In [18]:
def seed_everything(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

seed_everything(42)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

Using device: cpu


## Cell 4 — Load CSV files

In [19]:
train_df = pd.read_csv("train.csv")
val_df = pd.read_csv("val.csv")
test_df = pd.read_csv("test.csv")
sample_sub = pd.read_csv("sample_submission.csv")

print("Train shape:", train_df.shape)
print("Val shape:", val_df.shape)
print("Test shape:", test_df.shape)

display(train_df.head())
display(val_df.head())
display(test_df.head())

Train shape: (4099, 4)
Val shape: (878, 4)
Test shape: (879, 3)


,id,filepath,filename,target
0,CXR_05624,train/PNEUMONIA/CXR_05624.jpeg,CXR_05624.jpeg,1
1,CXR_01796,train/PNEUMONIA/CXR_01796.jpeg,CXR_01796.jpeg,1
2,CXR_04612,train/PNEUMONIA/CXR_04612.jpeg,CXR_04612.jpeg,1
3,CXR_02637,train/PNEUMONIA/CXR_02637.jpeg,CXR_02637.jpeg,1
4,CXR_02885,train/NORMAL/CXR_02885.jpeg,CXR_02885.jpeg,0


,id,filepath,filename,target
0,CXR_00918,val/NORMAL/CXR_00918.jpeg,CXR_00918.jpeg,0
1,CXR_04666,val/PNEUMONIA/CXR_04666.jpeg,CXR_04666.jpeg,1
2,CXR_01884,val/PNEUMONIA/CXR_01884.jpeg,CXR_01884.jpeg,1
3,CXR_00359,val/PNEUMONIA/CXR_00359.jpeg,CXR_00359.jpeg,1
4,CXR_01545,val/PNEUMONIA/CXR_01545.jpeg,CXR_01545.jpeg,1


,id,filepath,filename
0,CXR_05741,test/CXR_05741.jpeg,CXR_05741.jpeg
1,CXR_03646,test/CXR_03646.jpeg,CXR_03646.jpeg
2,CXR_00870,test/CXR_00870.jpeg,CXR_00870.jpeg
3,CXR_04969,test/CXR_04969.jpeg,CXR_04969.jpeg
4,CXR_01893,test/CXR_01893.jpeg,CXR_01893.jpeg


## Cell 5 — Basic integrity checks

In [20]:
print("Train class counts:")
print(train_df["target"].value_counts().sort_index())

print("\nVal class counts:")
print(val_df["target"].value_counts().sort_index())

print("\nDuplicate IDs in train:", train_df["id"].duplicated().sum())
print("Duplicate IDs in val:", val_df["id"].duplicated().sum())
print("Duplicate IDs in test:", test_df["id"].duplicated().sum())

Train class counts:
target
0    1108
1    2991
Name: count, dtype: int64

Val class counts:
target
0    237
1    641
Name: count, dtype: int64

Duplicate IDs in train: 0
Duplicate IDs in val: 0
Duplicate IDs in test: 0


## Cell 6 — Define transforms

In [21]:
train_transforms = transforms.Compose([
    transforms.Resize((128, 128)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(7),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.5], std=[0.5])
])

val_test_transforms = transforms.Compose([
    transforms.Resize((128, 128)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.5], std=[0.5])
])

## Cell 7 — Custom dataset

In [22]:
class ChestXRayDataset(Dataset):
    def __init__(self, df, transform=None):
        self.df = df.reset_index(drop=True)
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        img_path = self.df.loc[idx, "filepath"]
        image = Image.open(img_path).convert("L")  # grayscale

        if self.transform:
            image = self.transform(image)

        if "target" in self.df.columns:
            label = torch.tensor(self.df.loc[idx, "target"], dtype=torch.float32)
            return image, label
        else:
            image_id = self.df.loc[idx, "id"]
            return image, image_id

## Cell 8 — Weighted sampler for class imbalance

In [23]:
class_counts = train_df["target"].value_counts().sort_index().values
print("Class counts [Normal, Pneumonia]:", class_counts)

class_weights = 1.0 / class_counts
print("Class weights:", class_weights)

sample_weights = train_df["target"].map({
    0: class_weights[0],
    1: class_weights[1]
}).values

sample_weights = torch.DoubleTensor(sample_weights)

train_sampler = WeightedRandomSampler(
    weights=sample_weights,
    num_samples=len(sample_weights),
    replacement=True
)

Class counts [Normal, Pneumonia]: [1108 2991]
Class weights: [0.00090253 0.00033434]


C:\Users\BLACK\AppData\Local\Temp\ipykernel_1788\3110020648.py:12: UserWarning: The given NumPy array is not writable, and PyTorch does not support non-writable tensors. This means writing to this tensor will result in undefined behavior. You may want to copy the array to protect its data or make it writable before converting it to a tensor. This type of warning will be suppressed for the rest of this program. (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\pytorch\torch\csrc\utils\tensor_numpy.cpp:219.)
  sample_weights = torch.DoubleTensor(sample_weights)


## Cell 9 — DataLoaders

In [24]:
use_cuda = torch.cuda.is_available()
BATCH_SIZE = 32

train_dataset = ChestXRayDataset(train_df, transform=train_transforms)
val_dataset = ChestXRayDataset(val_df, transform=val_test_transforms)
test_dataset = ChestXRayDataset(test_df, transform=val_test_transforms)

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    sampler=train_sampler,
    num_workers=0,
    pin_memory=use_cuda
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=0,
    pin_memory=use_cuda
)

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=0,
    pin_memory=use_cuda
)

print(f"Using CUDA: {use_cuda}")
print("Train batches:", len(train_loader))
print("Val batches:", len(val_loader))
print("Test batches:", len(test_loader))

Using CUDA: False
Train batches: 129
Val batches: 28
Test batches: 28


## Cell 10 — CNN architecture

In [25]:
class PneumoniaCNN(nn.Module):
    def __init__(self):
        super().__init__()

        def conv_block(in_channels, out_channels):
            return nn.Sequential(
                nn.Conv2d(in_channels, out_channels, kernel_size=3, padding=1),
                nn.BatchNorm2d(out_channels),
                nn.ReLU(inplace=True),
                nn.MaxPool2d(2)
            )

        self.features = nn.Sequential(
            conv_block(1, 16),    # 128 -> 64
            conv_block(16, 32),   # 64 -> 32
            conv_block(32, 64),   # 32 -> 16
            conv_block(64, 128)   # 16 -> 8
        )

        self.classifier = nn.Sequential(
            nn.AdaptiveAvgPool2d(1),
            nn.Flatten(),
            nn.Dropout(0.3),
            nn.Linear(128, 32),
            nn.ReLU(inplace=True),
            nn.Dropout(0.2),
            nn.Linear(32, 1)
        )

    def forward(self, x):
        x = self.features(x)
        x = self.classifier(x)
        return x

model = PneumoniaCNN().to(device)
print(model)

PneumoniaCNN(
  (features): Sequential(
    (0): Sequential(
      (0): Conv2d(1, 16, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
      (1): BatchNorm2d(16, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (2): ReLU(inplace=True)
      (3): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    )
    (1): Sequential(
      (0): Conv2d(16, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
      (1): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (2): ReLU(inplace=True)
      (3): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    )
    (2): Sequential(
      (0): Conv2d(32, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
      (1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (2): ReLU(inplace=True)
      (3): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    )
    (3): Sequential(
      (

## Cell 11 — Loss, optimizer, scheduler

In [26]:
criterion = nn.BCEWithLogitsLoss()

optimizer = optim.Adam(model.parameters(), lr=1e-3)

scheduler = optim.lr_scheduler.ReduceLROnPlateau(
    optimizer,
    mode="max",
    factor=0.1,
    patience=2
)

## Cell 12 — Training function

In [27]:
def train_one_epoch(model, loader, criterion, optimizer, device):
    model.train()
    running_loss = 0.0

    for images, labels in loader:
        images = images.to(device)
        labels = labels.to(device).unsqueeze(1)

        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item()

    epoch_loss = running_loss / len(loader)
    return epoch_loss

## Cell 13 — Evaluation function

In [28]:
def evaluate(model, loader, device, threshold=0.5):
    model.eval()

    y_true = []
    y_pred = []
    y_prob = []

    with torch.no_grad():
        for images, labels in loader:
            images = images.to(device)
            outputs = model(images)

            probs = torch.sigmoid(outputs).cpu().numpy().ravel()
            preds = (probs >= threshold).astype(int)
            true = labels.numpy().astype(int).ravel()

            y_prob.extend(probs.tolist())
            y_pred.extend(preds.tolist())
            y_true.extend(true.tolist())

    macro_f1 = f1_score(y_true, y_pred, average="macro")
    return macro_f1, y_true, y_pred, y_prob

## Cell 14 — Threshold tuning

In [29]:
def find_best_threshold(model, loader, device):
    model.eval()

    y_true = []
    y_prob = []

    with torch.no_grad():
        for images, labels in loader:
            images = images.to(device)
            outputs = model(images)

            probs = torch.sigmoid(outputs).cpu().numpy().ravel()
            true = labels.numpy().astype(int).ravel()

            y_prob.extend(probs.tolist())
            y_true.extend(true.tolist())

    best_threshold = 0.5
    best_f1 = 0.0

    for threshold in np.arange(0.10, 0.91, 0.05):
        preds = (np.array(y_prob) >= threshold).astype(int)
        score = f1_score(y_true, preds, average="macro")

        if score > best_f1:
            best_f1 = score
            best_threshold = threshold

    return best_threshold, best_f1

## Cell 15 — Training loop with best model saving

In [30]:
EPOCHS = 15
best_val_f1 = 0.0
best_epoch = 0

history = {
    "train_loss": [],
    "val_f1": []
}

for epoch in range(1, EPOCHS + 1):
    train_loss = train_one_epoch(model, train_loader, criterion, optimizer, device)
    val_f1, _, _, _ = evaluate(model, val_loader, device, threshold=0.5)

    scheduler.step(val_f1)

    history["train_loss"].append(train_loss)
    history["val_f1"].append(val_f1)

    if val_f1 > best_val_f1:
        best_val_f1 = val_f1
        best_epoch = epoch
        torch.save(model.state_dict(), "best_pneumonia_cnn.pt")

    print(
        f"Epoch {epoch:02d} | "
        f"Train Loss: {train_loss:.4f} | "
        f"Val Macro-F1: {val_f1:.4f}"
    )

print(f"\nBest epoch: {best_epoch}")
print(f"Best validation Macro-F1 @0.5 threshold: {best_val_f1:.4f}")

Epoch 01 | Train Loss: 0.3568 | Val Macro-F1: 0.5935
Epoch 02 | Train Loss: 0.2581 | Val Macro-F1: 0.9118
Epoch 03 | Train Loss: 0.2302 | Val Macro-F1: 0.7251
Epoch 04 | Train Loss: 0.2016 | Val Macro-F1: 0.8816
Epoch 05 | Train Loss: 0.2147 | Val Macro-F1: 0.9084
Epoch 06 | Train Loss: 0.1746 | Val Macro-F1: 0.9332
Epoch 07 | Train Loss: 0.1712 | Val Macro-F1: 0.9327
Epoch 08 | Train Loss: 0.1658 | Val Macro-F1: 0.9352
Epoch 09 | Train Loss: 0.1667 | Val Macro-F1: 0.9224
Epoch 10 | Train Loss: 0.1793 | Val Macro-F1: 0.9359
Epoch 11 | Train Loss: 0.1421 | Val Macro-F1: 0.9383
Epoch 12 | Train Loss: 0.1524 | Val Macro-F1: 0.9396
Epoch 13 | Train Loss: 0.1581 | Val Macro-F1: 0.9221
Epoch 14 | Train Loss: 0.1587 | Val Macro-F1: 0.9372
Epoch 15 | Train Loss: 0.1665 | Val Macro-F1: 0.9336

Best epoch: 12
Best validation Macro-F1 @0.5 threshold: 0.9396


## Cell 16 — Load best model

In [31]:
model.load_state_dict(torch.load("best_pneumonia_cnn.pt", map_location=device))
model.eval()
print("Best model loaded.")

Best model loaded.


## Cell 17 — Tune threshold on validation set

In [32]:
best_threshold, tuned_f1 = find_best_threshold(model, val_loader, device)

print("Best threshold:", best_threshold)
print("Best validation Macro-F1 after threshold tuning:", tuned_f1)

Best threshold: 0.30000000000000004
Best validation Macro-F1 after threshold tuning: 0.9444987856406162


## Cell 18 — Final validation metrics

In [33]:
from sklearn.metrics import recall_score

final_f1, y_true, y_pred, y_prob = evaluate(
    model,
    val_loader,
    device,
    threshold=best_threshold
)

pneumonia_recall = recall_score(y_true, y_pred, pos_label=1)
normal_recall = recall_score(y_true, y_pred, pos_label=0)

print("Final Validation Macro-F1:", final_f1)
print(f"Pneumonia Recall: {pneumonia_recall:.4f}")
print(f"Normal Recall:    {normal_recall:.4f}")

print("\nConfusion Matrix:")
print(confusion_matrix(y_true, y_pred))

print("\nClassification Report:")
print(classification_report(y_true, y_pred, digits=4))

Final Validation Macro-F1: 0.9444987856406162
Pneumonia Recall: 0.9766
Normal Recall:    0.9030

Confusion Matrix:
[[214  23]
 [ 15 626]]

Classification Report:
              precision    recall  f1-score   support

           0     0.9345    0.9030    0.9185       237
           1     0.9646    0.9766    0.9705       641

    accuracy                         0.9567       878
   macro avg     0.9495    0.9398    0.9445       878
weighted avg     0.9564    0.9567    0.9565       878



## Cell 19 — Generate test predictions

In [34]:
model.eval()
results = []

with torch.no_grad():
    for images, image_ids in test_loader:
        images = images.to(device)
        outputs = model(images)

        probs = torch.sigmoid(outputs).cpu().numpy().ravel()
        preds = (probs >= best_threshold).astype(int)

        for image_id, pred in zip(image_ids, preds):
            results.append({
                "id": image_id,
                "target": int(pred)
            })

submission = pd.DataFrame(results)
submission.head()

,id,target
0,CXR_05741,1
1,CXR_03646,1
2,CXR_00870,1
3,CXR_04969,1
4,CXR_01893,1


## Cell 20 — Validate submission format

In [35]:
print("Submission shape:", submission.shape)
print("Expected shape:", sample_sub.shape)

print("\nSubmission columns:", submission.columns.tolist())
print("Expected columns:", sample_sub.columns.tolist())

print("\nMissing IDs:", set(sample_sub["id"]) - set(submission["id"]))
print("Extra IDs:", set(submission["id"]) - set(sample_sub["id"]))

Submission shape: (879, 2)
Expected shape: (879, 2)

Submission columns: ['id', 'target']
Expected columns: ['id', 'target']

Missing IDs: set()
Extra IDs: set()


## Cell 21 — Save submission file

In [38]:
submission = submission[["id", "target"]]
submission.to_csv("Team_MedAi_Zambia_submission1.csv", index=False)

print("Submission file saved as Team_MedAi_Zambia_submission1.csv")

Submission file saved as Team_MedAi_Zambia_submission1.csv


## Cell 22 — Optional notes for summary PDF

In [37]:
# Run threshold tuning FIRST
best_threshold, tuned_f1 = find_best_threshold(model, val_loader, device)

# Build dynamic summary
summary_notes = f"""

Architecture:
- Custom CNN built from scratch using PyTorch (torch.nn)
- 5 convolutional blocks: Conv → BatchNorm → ReLU → MaxPooling
- Feature maps increase from 1 → 256 channels
- Adaptive average pooling followed by fully connected classifier
- Dropout used for regularization (0.4, 0.2)
- Single-output logit for binary classification (sigmoid applied during inference)

Preprocessing:
- Images loaded via CSV metadata (filepath-based)
- Converted to grayscale (single-channel input)
- Resized to 128x128 pixels
- Normalized pixel intensities
- Light data augmentation: horizontal flip and small-angle rotation

Imbalance handling:
- WeightedRandomSampler used to address class imbalance between Normal and Pneumonia classes

Training:
- Loss function: BCEWithLogitsLoss
- Optimizer: Adam (lr=1e-3)
- Learning rate scheduling: ReduceLROnPlateau (monitoring validation macro-F1)
- Training performed for 15 epochs
- Best model checkpoint selected based on highest validation macro-F1 at threshold = 0.5

Evaluation:
- Primary metric: macro-averaged F1-score
- Threshold tuning performed on validation probabilities (range: 0.10–0.90)
- Final evaluation performed using optimized threshold

Current Best Validation Performance:
- Best epoch: {best_epoch}
- Macro-F1 (@0.5 threshold): {best_val_f1:.4f}
- Optimal threshold: {best_threshold:.2f}
- Final Macro-F1 after threshold tuning: {tuned_f1:.4f}

"""

print(summary_notes)



Architecture:
- Custom CNN built from scratch using PyTorch (torch.nn)
- 5 convolutional blocks: Conv → BatchNorm → ReLU → MaxPooling
- Feature maps increase from 1 → 256 channels
- Adaptive average pooling followed by fully connected classifier
- Dropout used for regularization (0.4, 0.2)
- Single-output logit for binary classification (sigmoid applied during inference)

Preprocessing:
- Images loaded via CSV metadata (filepath-based)
- Converted to grayscale (single-channel input)
- Resized to 128x128 pixels
- Normalized pixel intensities
- Light data augmentation: horizontal flip and small-angle rotation

Imbalance handling:
- WeightedRandomSampler used to address class imbalance between Normal and Pneumonia classes

Training:
- Loss function: BCEWithLogitsLoss
- Optimizer: Adam (lr=1e-3)
- Learning rate scheduling: ReduceLROnPlateau (monitoring validation macro-F1)
- Training performed for 15 epochs
- Best model checkpoint selected based on highest validation macro-F1 at threshol